In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.manifold import TSNE
import umap

datasets = {
    "full": "../../data/processed/ProcessedData.csv",
    "pca": "../../data/reduced/dataPCA.csv",
    "fs": "../../data/reduced/dataFS.csv"
}

outDir = "../../results/gmm"

In [2]:
def plotGMM(X, labels, method, name):
    plt.figure(figsize=(6,5))
    plt.scatter(X[:,0], X[:,1], c=labels, cmap="tab10", s=5)
    plt.title(f"GMM {method} ({name})")
    plt.tight_layout()
    plt.savefig(f"{outDir}/gmm_{method}_{name}.png", dpi=200)
    plt.close()

In [3]:
def select_gmm_components(X, max_components=10):
    bics = []
    models = []
    
    for k in range(2, max_components+1):
        gmm = GaussianMixture(n_components=k, covariance_type="full", random_state=42)
        gmm.fit(X)
        bics.append(gmm.bic(X))
        models.append(gmm)
    
    best_idx = np.argmin(bics)
    return models[best_idx], best_idx + 2, bics

In [4]:
results = []

for name, path in datasets.items():
    
    df = pd.read_csv(path)

    if "Class" in df.columns:
        y_true = df["Class"].values
        X = df.drop(columns=["Class"])
    else:
        y_true = None
        X = df.copy()

    if "V129" in X.columns:
        X = X.drop(columns=["V129"])

    X_np = X.values

    best_gmm, best_k, bics = select_gmm_components(X_np, max_components=10)
    labels = best_gmm.predict(X_np)

    unique_clusters = len(set(labels))

    if unique_clusters > 1:
        sil = silhouette_score(X_np, labels)
        db  = davies_bouldin_score(X_np, labels)
    else:
        sil, db = np.nan, np.nan

    if y_true is not None:
        ari = adjusted_rand_score(y_true, labels)
        nmi = normalized_mutual_info_score(y_true, labels)
    else:
        ari, nmi = np.nan, np.nan

    out = X.copy()
    out["cluster"] = labels
    out.to_csv(f"{outDir}/gmm_{name}_clusters.csv", index=False)

    reducer = umap.UMAP(n_components=2, random_state=42)
    X_umap = reducer.fit_transform(X_np)
    plotGMM(X_umap, labels, "UMAP", name)

    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    X_tsne = tsne.fit_transform(X_np)
    plotGMM(X_tsne, labels, "TSNE", name)

    results.append({
        "Dataset": name,
        "Algorithm": "GMM",
        "BestComponents": best_k,
        "Silhouette": sil,
        "Davies_Bouldin": db,
        "ARI": ari,
        "NMI": nmi
    })

C:\Users\Korisnik\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\Korisnik\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\Korisnik\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [5]:
summaryData = pd.DataFrame(results)
summaryData.to_csv(f"{outDir}/gmm_summary.csv", index=False)
print(summaryData)

  Dataset Algorithm  BestComponents  Silhouette  Davies_Bouldin       ARI  \
0    full       GMM               9    0.095714        1.673806  0.211281   
1     pca       GMM              10    0.011954        2.612316       NaN   
2      fs       GMM               9    0.006518        2.305347       NaN   

        NMI  
0  0.365913  
1       NaN  
2       NaN  
